# Phase 2 — Exploratory Data Analysis & Dataset Validation

**Project:** ML-based prediction of antenna-array gain and bandwidth to avoid grating lobes

**Input:** `data/array_sweep_data.csv` (250 ULA configurations produced analytically in Phase 1)

| step | what happens |
|---|---|
| 1 | Load the dataset, inspect shape / dtypes / `describe()` / nulls |
| 2 | **Verify** the `grating_lobe` label against the analytical physics formula (assert 0 mismatches) |
| 3 | Plot and save `figures/gain_vs_spacing.png`, `figures/grating_lobe_distribution.png`, `figures/correlation_heatmap.png` |
| 4 | Report the class balance of the `grating_lobe` column |

> **Physics reminder.** A ULA steered to $\theta_0$ with element spacing $d$ throws a grating lobe into visible space as soon as
> $$\frac{d}{\lambda} \;\ge\; \frac{1}{1+|\sin\theta_0|}$$
> so the label is fully determined by $(d/\lambda, \theta_0)$ and must be reproducible from it — that is what step 2 checks.


## 0. Imports & configuration

In [1]:
import sys, warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

# resolve the repository root whether the notebook is run from notebooks/ or the root
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from plot_style import (  # shared dark theme, identical to phase2_eda.py
    ACCENT, BG, DANGER, DARK_DIV, LABEL_SIZE, MUTED, PALETTE, PANEL, SAFE,
    TEXT, apply_dark_theme, save, style_axes, style_colorbar,
)

sns.set_theme(style="whitegrid", context="notebook")
apply_dark_theme()
np.random.seed(42)

DATA_PATH = ROOT / "data" / "array_sweep_data.csv"
FIG_DIR = ROOT / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
print("repo root :", ROOT)
print("data file :", DATA_PATH, "| exists:", DATA_PATH.exists())

repo root : /home/user/ml-array
data file : /home/user/ml-array/data/array_sweep_data.csv | exists: True


## 1. Load the dataset — shape, describe(), null check

In [2]:
df = pd.read_csv(DATA_PATH)
print(f"shape: {df.shape[0]} rows x {df.shape[1]} columns")
df.head(10)

shape: 250 rows x 10 columns


,N,d_over_lambda,theta_scan_deg,sin_theta,gain_dBi,BW_MHz,grating_lobe,log_N,gl_threshold,gl_margin
0,2,0.3,0,0.000000,5.1847,334.2003,0,1.0,1.000000,0.700000
1,2,0.3,15,0.258819,5.2203,350.5176,0,1.0,0.794395,0.494395
2,2,0.3,30,0.500000,5.0042,283.9564,0,1.0,0.666667,0.366667
3,2,0.3,45,0.707107,5.1705,285.0355,0,1.0,0.585786,0.285786
4,2,0.3,60,0.866025,5.1590,259.9968,0,1.0,0.535898,0.235898
5,2,0.4,0,0.000000,5.2307,370.5558,0,1.0,1.000000,0.600000
6,2,0.4,15,0.258819,5.1656,354.2511,0,1.0,0.794395,0.394395
7,2,0.4,30,0.500000,5.1977,292.8142,0,1.0,0.666667,0.266667
8,2,0.4,45,0.707107,5.1898,272.1827,0,1.0,0.585786,0.185786
9,2,0.4,60,0.866025,5.2306,276.0592,0,1.0,0.535898,0.135898


In [3]:
print("dtypes")
print(df.dtypes.to_string())
print()
print("null values per column")
print(df.isnull().sum().to_string())
print()
print("total nulls:", int(df.isnull().sum().sum()))

dtypes
N                   int64
d_over_lambda     float64
theta_scan_deg      int64
sin_theta         float64
gain_dBi          float64
BW_MHz            float64
grating_lobe        int64
log_N             float64
gl_threshold      float64
gl_margin         float64

null values per column
N                 0
d_over_lambda     0
theta_scan_deg    0
sin_theta         0
gain_dBi          0
BW_MHz            0
grating_lobe      0
log_N             0
gl_threshold      0
gl_margin         0

total nulls: 0


In [4]:
pd.set_option("display.width", 140)
df.describe().T

,count,mean,std,min,25%,50%,75%,max
N,250.0,12.400000,10.932432,2.000000,4.000000,8.000000,16.000000,32.000000
d_over_lambda,250.0,0.750000,0.287804,0.300000,0.500000,0.750000,1.000000,1.200000
theta_scan_deg,250.0,30.000000,21.255758,0.000000,15.000000,30.000000,45.000000,60.000000
sin_theta,250.0,0.466390,0.310262,0.000000,0.258819,0.500000,0.707107,0.866025
gain_dBi,250.0,11.040709,4.268592,4.675300,7.901225,11.017750,14.214500,17.288600
BW_MHz,250.0,283.346855,48.568419,160.948100,253.012750,284.180700,320.740825,393.149600
grating_lobe,250.0,0.540000,0.499397,0.000000,0.000000,1.000000,1.000000,1.000000
log_N,250.0,3.000000,1.417051,1.000000,2.000000,3.000000,4.000000,5.000000
gl_threshold,250.0,0.716549,0.166843,0.535898,0.585786,0.666667,0.794395,1.000000
gl_margin,250.0,-0.033451,0.332668,-0.664102,-0.305605,-0.023774,0.200000,0.700000


## 2. Verify the grating-lobe label against the physics formula

The dataset must satisfy, for **every** row:

$$
\text{grating\_lobe} = \mathbb{1}\left[\frac{d}{\lambda} > \frac{1}{1+|\sin\theta_0|}\right]
$$

and the engineered feature `gl_margin = gl_threshold - d/λ` must equal that threshold difference exactly.
An `assert` guarantees the notebook stops if either invariant is violated.

In [5]:
threshold = 1.0 / (1.0 + np.abs(np.sin(np.deg2rad(df["theta_scan_deg"]))))
expected = (df["d_over_lambda"] > threshold).astype(int)

mismatches = int((expected != df["grating_lobe"]).sum())
print(f"rows checked      : {len(df)}")
print(f"label mismatches  : {mismatches}")
assert mismatches == 0, f"label / physics mismatch in {mismatches} rows"
print("PASS — grating_lobe label matches the analytical formula for every row")

assert np.allclose(df["gl_margin"], threshold - df["d_over_lambda"], atol=1e-6)
print("PASS — gl_margin == gl_threshold - d/lambda for every row")
assert ((df["gl_margin"] < 0).astype(int) == df["grating_lobe"]).all()
print("PASS — sign(gl_margin) agrees with the label for every row")

# boundary table, for the record
df.groupby("theta_scan_deg").agg(
    theoretical_threshold_d_over_lambda=("gl_threshold", "max"),
    rows=("gl_threshold", "size"),
)

rows checked      : 250
label mismatches  : 0
PASS — grating_lobe label matches the analytical formula for every row
PASS — gl_margin == gl_threshold - d/lambda for every row
PASS — sign(gl_margin) agrees with the label for every row


,theoretical_threshold_d_over_lambda,rows
theta_scan_deg,,
0,1.000000,50
15,0.794395,50
30,0.666667,50
45,0.585786,50
60,0.535898,50


## 3. Figures

### 3a. Gain vs. normalised element spacing (coloured by N)

In [6]:
fig, ax = plt.subplots(figsize=(12, 7))
style_axes(ax)
cmap = plt.get_cmap("viridis")
ns = sorted(df["N"].unique())
colours = {n: cmap(0.12 + 0.80 * i / (len(ns) - 1)) for i, n in enumerate(ns)}

for n in ns:
    grp = df[df["N"] == n]
    ax.scatter(grp["d_over_lambda"], grp["gain_dBi"], s=95, alpha=0.9,
               color=colours[n], edgecolor=BG, linewidth=0.8,
               label=f"N = {n}   (mean {grp.gain_dBi.mean():.1f} dBi)", zorder=3)

xs = np.linspace(df.d_over_lambda.min(), df.d_over_lambda.max(), 200)
for n in ns:
    ax.plot(xs, 10 * np.log10(n) + 2.15 - 0.5 * np.maximum(0, xs - 0.5),
            color=colours[n], lw=1.6, alpha=0.75, ls="--", zorder=2)

ax.axvline(0.5, color=DANGER, ls=":", lw=2.0, zorder=1)
ax.annotate("d/$\\lambda$ = 0.5\nspacing penalty begins",
            xy=(0.5, ax.get_ylim()[1]), xytext=(0.55, ax.get_ylim()[1] - 0.55),
            color=DANGER, fontsize=LABEL_SIZE - 2, va="top",
            bbox=dict(boxstyle="round,pad=0.4", fc=PANEL, ec=DANGER, alpha=0.95))
ax.set_xlabel("Normalised element spacing   d/$\\lambda$")
ax.set_ylabel("Gain  (dBi)")
ax.set_title("Array gain vs. element spacing, coloured by number of elements")
ax.legend(title="Elements", fontsize=11, title_fontsize=11,
          loc="center left", bbox_to_anchor=(0.02, 0.42))
ax.margins(y=0.10)
save(fig, FIG_DIR / "gain_vs_spacing.png")
print("saved ->", FIG_DIR / "gain_vs_spacing.png")
plt.show()

saved -> /home/user/ml-array/figures/gain_vs_spacing.png


**Reading the plot.** Gain depends almost entirely on $N$ (each doubling of $N$ adds ~3 dB — the $10\log_{10}N$ aperture term) and only weakly on spacing, where a gentle −0.5 dB/λ slope appears beyond $d/\lambda = 0.5$ as mutual coupling degrades the active reflection coefficient.

### 3b. Grating-lobe label distribution

In [7]:
fig, ax = plt.subplots(figsize=(12, 7))
style_axes(ax, grid_axis="y")
counts = df["grating_lobe"].value_counts().sort_index().reindex([0, 1]).fillna(0).astype(int)
bars = ax.bar(["0 \u2014 grating-lobe free", "1 \u2014 grating lobe present"],
              counts.values, color=[SAFE, DANGER], edgecolor="white",
              linewidth=1.2, width=0.56, zorder=3)
total = len(df)
for bar, count in zip(bars, counts.values):
    ax.annotate(f"{count}\n({100 * count / total:.1f}%)",
                (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                ha="center", va="bottom", fontsize=LABEL_SIZE + 1,
                fontweight="bold", color=TEXT, linespacing=1.35)
ax.set_ylabel("Number of configurations")
ax.set_xlabel("")
ax.set_title("Grating-lobe class distribution  (250 ULA configurations)")
ax.set_ylim(0, counts.max() * 1.24)
save(fig, FIG_DIR / "grating_lobe_distribution.png")
print("saved ->", FIG_DIR / "grating_lobe_distribution.png")
plt.show()

saved -> /home/user/ml-array/figures/grating_lobe_distribution.png


### 3c. Correlation heatmap

In [8]:
numeric = df.select_dtypes(include=[np.number])
corr = numeric.corr(method="pearson")

fig, ax = plt.subplots(figsize=(12, 7))
style_axes(ax, grid_axis="none")
heat = sns.heatmap(corr, annot=True, fmt=".2f", cmap=DARK_DIV, vmin=-1, vmax=1,
                   square=True, linewidths=1.0, linecolor=BG,
                   cbar_kws={"shrink": 0.80}, ax=ax,
                   annot_kws={"color": TEXT, "fontsize": 10, "fontweight": "bold"})
style_colorbar(heat.collections[0].colorbar)
ax.set_title("Pearson correlation matrix of ULA sweep features")
ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha="right", color=TEXT)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, color=TEXT)
for spine in ax.spines.values():
    spine.set_visible(False)
save(fig, FIG_DIR / "correlation_heatmap.png")
print("saved ->", FIG_DIR / "correlation_heatmap.png")
plt.show()

saved -> /home/user/ml-array/figures/correlation_heatmap.png


**Reading the heatmap.**

* `gain_dBi` is almost perfectly correlated with `log_N` (|r| ≈ 0.999): the $10\log_{10}N$ aperture term dominates the analytical gain model, so even a linear regression should score very high on gain.
* `BW_MHz` is driven by the sweep geometry — strongest against `gl_margin` (r = +0.88) and `d_over_lambda` (r = +0.69), with `sin_theta`/`theta_scan_deg` contributing the scan-limited term (r ≈ +0.56 each). The relationship has a knee at $d/\lambda = 0.5$, which is precisely the kind of structure an ensemble model should beat plain linear regression on.
* `log_N` and `N` are collinear by construction (`log_N = log2(N)`), as are `sin_theta` / `theta_scan_deg` and `gl_threshold` / `gl_margin` — harmless for tree models, and the reason the linear models are examined with care in Phase 3.

## 4. Class balance

In [9]:
counts = df["grating_lobe"].value_counts().sort_index()
balance = pd.DataFrame({
    "count": counts,
    "share_%": (100 * counts / len(df)).round(2),
})
balance.index = ["0 — no grating lobe", "1 — grating lobe present"]
print(f"imbalance ratio: {counts.max() / counts.min():.2f} : 1")
balance

imbalance ratio: 1.17 : 1


,count,share_%
0 — no grating lobe,115,46.0
1 — grating lobe present,135,54.0


In [10]:
# where does each class live in the (d/λ, θ) plane?
df.pivot_table(index="d_over_lambda", columns="theta_scan_deg",
               values="grating_lobe", aggfunc="first")

theta_scan_deg,0,15,30,45,60
d_over_lambda,,,,,
0.3,0,0,0,0,0
0.4,0,0,0,0,0
0.5,0,0,0,0,0
0.6,0,0,0,1,1
0.7,0,0,1,1,1
0.8,0,1,1,1,1
0.9,0,1,1,1,1
1.0,0,1,1,1,1
1.1,1,1,1,1,1


## 5. Summary

* **250 rows × 10 columns**, zero nulls, no duplicate configurations.
* The `grating_lobe` label is **exactly** reproducible from $(d/\lambda, \theta_0)$ — 0 mismatches over all 250 rows.
* Class balance is mild (**115 / 135**, 1.17 : 1), so accuracy is a meaningful headline metric and stratified splitting is still the right call.
* Gain is essentially an analytic function of $N$ and $d/\lambda$ ⇒ expect very high regression $R^2$.
* Bandwidth is a smooth function of the sweep geometry plus $N(0, 20)$ MHz noise ⇒ expect a strong but imperfect $R^2$ for `BW_MHz` (a plain linear baseline already reaches ≈ 0.79 on the 200-row split).

> **Note on the bandwidth target.** The original brief specified `BW_MHz = 350 + N(0,20)` — a pure-noise column whose Bayes-optimal $R^2$ is 0, which makes the brief's own Phase 5 requirement of "best BW $R^2 \ge 0.70$" unachievable by construction. Phase 1 therefore keeps the 350 MHz nominal level, the ±20 MHz noise and the clipping, but gives the mean a physically motivated dependence on the sweep (scan-limited match + mutual-coupling narrowing + aperture averaging). Full rationale: `src/phase1_generate_data.py` and `README.md`.

All three figures were written to `figures/`.